# 02 · Preprocess

Look at the masks, body keypoints and face landmarks written by `avatar-splats preprocess`. Run that first; this notebook only reads its results.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from avatar_splats.config import VIDEOS, load_config
from avatar_splats.data import read_frames
from avatar_splats.preprocess import Preprocessed, preprocessed_path

config = load_config("../configs/default.yaml")
results = {name: Preprocessed.load(preprocessed_path(config, name)) for name in VIDEOS}
frames = {
    name: list(read_frames(getattr(config.capture, name), step=config.preprocess.frame_step))
    for name in VIDEOS
}

## Overlays

Six frames per video: the mask tinted, its box, and the body keypoints or face landmarks. Pick another video with `name`.

In [ ]:
name = "body"
result, video = results[name], frames[name]

shown = np.linspace(0, len(video) - 1, 6).round().astype(int)
fig, axes = plt.subplots(1, len(shown), figsize=(3 * len(shown), 6))
for ax, i in zip(axes, shown, strict=True):
    tinted = video[i].copy()
    tinted[~result.masks[i]] //= 3
    ax.imshow(tinted)
    x0, y0, x1, y1 = result.boxes[i]
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="yellow"))
    if result.keypoints is not None:
        x, y, confidence = result.keypoints[i].T
        ax.scatter(x, y, c=confidence, cmap="RdYlGn", vmin=0, vmax=1, s=12)
    if result.face_landmarks is not None:
        ax.scatter(*result.face_landmarks[i, :, :2].T, s=0.5, c="cyan")
    ax.set_title(f"frame {result.frame_indices[i]}")
    ax.axis("off")
fig.tight_layout()

## Over time

Mask area should change smoothly as the performer turns; a spike is a tracking failure. Keypoint confidence dips where joints are hidden in side views, and the face landmarker loses the face when the back of the head faces the camera.

In [ ]:
fig, (area_ax, detail_ax) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for name, result in results.items():
    area_ax.plot(result.frame_indices, result.masks.mean(axis=(1, 2)), label=name)
    if result.keypoints is not None:
        confidence = result.keypoints[..., 2]
        detail_ax.plot(result.frame_indices, confidence.min(axis=1), label=f"{name}: weakest joint")
    if result.face_landmarks is not None:
        found = ~np.isnan(result.face_landmarks).any(axis=(1, 2))
        detail_ax.plot(result.frame_indices, found, drawstyle="steps-mid", label=f"{name}: face")
area_ax.set_ylabel("mask area (fraction)")
detail_ax.set_xlabel("frame")
for ax in (area_ax, detail_ax):
    ax.legend()
fig.tight_layout()